# MMBench / Newt — loading and visualizing environments

A guided tour of the environment side of [*Learning Massively Multitask World
Models for Continuous Control*](https://arxiv.org/abs/2511.19584)
(Hansen, Su & Wang) — the **MMBench** task suite and the `make_env()` factory
that Newt trains against.

This notebook does **not** train anything. It:

1. loads the MMBench task registry and breaks it down by domain,
2. builds a config the same way `train.py` does (hydra structured config + `parse_cfg`),
3. constructs a single-task environment and runs a random-action rollout,
4. renders frames and writes a video,
5. repeats the construction across one task per domain, to confirm every
   simulator backend in the install is live.

### Prerequisites

```bash
export MS_SKIP_ASSET_DOWNLOAD_PROMPT=1
export MUJOCO_GL=egl          # NOT osmesa -- there is no OSMesa on this system
source /tmp/yn030245/newt-venv/bin/activate
```

`MUJOCO_GL=egl` is load-bearing: the conda recipe supplies MuJoCo's GL backend
via `glew`/`mesalib`, which the uv venv does not have. EGL comes from the
system NVIDIA driver instead.

## 1. Setup

The repo is not an installed package — `tdmpc2/` is the import root, exactly as `train.py` assumes.

In [1]:
import os

# Must be set before mujoco / dm_control are imported.
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("MS_SKIP_ASSET_DOWNLOAD_PROMPT", "1")
os.environ.setdefault("LAZY_LEGACY_OP", "0")

import json
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

REPO = Path.home() / "newt"
sys.path.insert(0, str(REPO / "tdmpc2"))   # import root for config/, envs/, common/

import numpy as np
import torch
import matplotlib.pyplot as plt

import mujoco
import gymnasium
import mani_skill

print(f"python      {sys.version.split()[0]}")
print(f"torch       {torch.__version__}   cuda={torch.cuda.is_available()}  "
      f"devices={torch.cuda.device_count()}")
print(f"mujoco      {mujoco.__version__}   MUJOCO_GL={os.environ['MUJOCO_GL']}")
print(f"gymnasium   {gymnasium.__version__}")
print(f"mani_skill  {mani_skill.__version__}")
print(f"numpy       {np.__version__}")

python      3.10.18
torch       2.8.0+cu128   cuda=True  devices=4
mujoco      3.3.6   MUJOCO_GL=egl
gymnasium   0.29.1
mani_skill  3.0.0b21
numpy       1.24.4


## 2. The MMBench task registry

`tdmpc2/common/__init__.py` defines `TASK_SET`, which maps a *task-set name* to
a list of task names. `"soup"` is the full 200-task multitask set Newt is
trained on. `tasks.json` additionally carries per-task metadata.

Task names are prefixed by domain — `mw-` Meta-World, `ms-` ManiSkill,
`og-` OGBench, `rd-` RoboDesk, `atari-`, `pygame-` (MiniArcade),
`mujoco-`, `bipedal-walker-`/`lunarlander-` (Box2D), and bare names for
DMControl.

In [2]:
from common import TASK_SET, MODEL_SIZE

tasks = TASK_SET["soup"]
print(f"tasks in 'soup': {len(tasks)}   (unique: {len(set(tasks))})")
print(f"other task sets: {[k for k in TASK_SET if k != 'soup']}")
print(f"model sizes:     {list(MODEL_SIZE)}")

PREFIX_TO_DOMAIN = {
    "mw-": "Meta-World", "ms-": "ManiSkill", "og-": "OGBench",
    "rd-": "RoboDesk", "atari-": "Atari", "pygame-": "MiniArcade",
    "mujoco-": "Gym-MuJoCo", "bipedal-walker-": "Box2D", "lunarlander-": "Box2D",
}

def domain_of(task):
    for p, d in PREFIX_TO_DOMAIN.items():
        if task.startswith(p):
            return d
    return "DMControl"

from collections import Counter
counts = Counter(domain_of(t) for t in dict.fromkeys(tasks))
for d, n in counts.most_common():
    print(f"  {d:<12} {n:>3}")

tasks in 'soup': 200   (unique: 200)
other task sets: ['dmcontrol', 'dmcontrol-ext', 'metaworld', 'maniskill', 'mujoco', 'box2d', 'robodesk', 'ogbench', 'pygame', 'atari']
model sizes:     ['S', 'B', 'L', 'XL']
  Meta-World    49
  DMControl     37
  ManiSkill     36
  Atari         27
  MiniArcade    19
  OGBench       12
  Box2D          8
  Gym-MuJoCo     6
  RoboDesk       6


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6), layout="constrained")

order = counts.most_common()
labels = [d for d, _ in order]
vals = [n for _, n in order]
bars = ax.barh(labels[::-1], vals[::-1], color="#4C72B0", height=0.68)
ax.bar_label(bars, padding=3, fontsize=9)

ax.set_xlabel("number of tasks")
ax.set_title(f"MMBench task suite by domain ({sum(vals)} tasks in 'soup')")
ax.set_xlim(0, max(vals) * 1.15)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="y", length=0)
fig.savefig("mmbench_domains.png", dpi=150)
plt.show()

## 3. Building a config the way `train.py` does

`Config` in `config.py` is a **hydra structured config** (a dataclass), not a
YAML file, and `parse_cfg()` fills in the derived fields — task list, model
dimensions for the chosen `model_size`, episode lengths, eval cadence.

`parse_cfg()` calls `hydra.utils.get_original_cwd()`, so it needs an active
hydra context. `train.py` gets that from its `@hydra.main` decorator; in a
notebook we set one up explicitly with `initialize` + `compose`.

In [3]:
from hydra import compose, initialize
from hydra.core.config_store import ConfigStore
from hydra.core.global_hydra import GlobalHydra
from hydra.core.hydra_config import HydraConfig
from omegaconf import OmegaConf

from config import Config, parse_cfg

ConfigStore.instance().store(name="config", node=Config)


def build_cfg(task, **overrides):
    """Compose + parse a Newt config outside of @hydra.main."""
    GlobalHydra.instance().clear()          # compose() is single-shot per process
    with initialize(version_base=None, config_path=None):
        cfg = compose(config_name="config", return_hydra_config=True)
    HydraConfig.instance().set_config(cfg)   # what get_original_cwd() reads
    OmegaConf.set_struct(cfg, False)
    cfg.pop("hydra", None)

    cfg.task = task
    cfg.num_envs = 1
    cfg.env_mode = "sync"                    # sync -> sub-envs reachable for render()
    cfg.obs = "state"
    for k, v in overrides.items():
        cfg[k] = v
    return parse_cfg(cfg)


cfg = build_cfg("walker-walk")
print(f"task        {cfg.task}")
print(f"model_size  {cfg.model_size}  (parse_cfg defaults single-task to 'B' = 5M)")
print(f"latent_dim  {cfg.latent_dim}")
print(f"num_tasks   {cfg.num_tasks}     num_envs {cfg.num_envs}")
print(f"obs         {cfg.obs}      render_size {cfg.render_size}")
print(f"use_demos   {cfg.use_demos}   task_dim {cfg.task_dim}")

task        walker-walk
model_size  B  (parse_cfg defaults single-task to 'B' = 5M)
latent_dim  512
num_tasks   1     num_envs 1
obs         state      render_size 224
use_demos   False   task_dim 0


In [4]:
cfg

Config(task='walker-walk', obs='state', num_envs=1, env_mode='sync', checkpoint=None, eval_episodes=2, steps=100000000, batch_size=1024, utd=0.075, reward_coef=0.1, value_coef=0.1, consistency_coef=20.0, prior_coef=10.0, rho=0.5, lr=0.0003, enc_lr_scale=0.3, grad_clip_norm=20.0, tau=0.01, discount_denom=5, discount_min=0.95, discount_max=0.995, buffer_size=10000000, use_demos=False, demo_steps=200000, lr_schedule=None, warmup_steps=5000, seeding_coef=5, exp_name='default', finetune=False, mpc=True, iterations=6, num_samples=512, num_elites=64, num_pi_trajs=24, horizon=3, min_std=0.05, max_std=2.0, temperature=0.5, constrained_planning=True, constraint_start_step=2000000, constraint_final_step=10000000, log_std_min=-10.0, log_std_max=2.0, entropy_coef=0.0001, num_bins=101, vmin=-10.0, vmax=10.0, model_size='B', num_enc_layers=2, enc_dim=256, mlp_dim=512, latent_dim=512, task_dim=0, num_q=5, simnorm_dim=8, wandb_project='<project>', wandb_entity='<user>', enable_wandb=True, multiproc=Fal

## 4. Constructing the environment

`make_env(cfg)` returns a **vectorized multitask wrapper**, even for a single
task — so observations and actions are always batched over `num_envs`, and
padded to MMBench's shared `MAX_OBS_DIM` / `MAX_ACTION_DIM`. That shared,
padded action space is what lets one Newt policy act in all 200 tasks.

In [ ]:
from envs import make_env

env = make_env(cfg)

obs, info = env.reset()
print(f"\nobs               {tuple(obs.shape)}  {obs.dtype}   (padded to MAX_OBS_DIM)")
print(f"action_space      {env.action_space.shape}       (padded to MAX_ACTION_DIM)")
print(f"true action dim   {cfg.action_dims}")
print(f"episode length    {env.max_episode_steps}")
print(f"info keys         {sorted(info.keys())}")

## 5. A random-action rollout

`rand_act()` samples uniformly in `[-1, 1]` over the padded action space; the
wrapper slices off each task's real action dimension.

In [ ]:
N_STEPS = 200
RENDER_EVERY = 4

rewards, frames = [], []
obs, info = env.reset()

for t in range(N_STEPS):
    action = env.rand_act()
    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(float(reward.mean()))
    if t % RENDER_EVERY == 0:
        frame = env.render(width=192, height=192)
        if frame is not None:
            frames.append(np.asarray(frame))

rewards = np.array(rewards)
print(f"steps            {len(rewards)}")
print(f"frames captured  {len(frames)}   shape {frames[0].shape}")
print(f"reward  mean {rewards.mean():.4f}   min {rewards.min():.4f}   "
      f"max {rewards.max():.4f}   sum {rewards.sum():.2f}")

In [ ]:
fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(7.5, 5.0), sharex=True,
                               height_ratios=[2, 1], layout="constrained")

ax0.plot(rewards, lw=1.0, color="#4C72B0", label="per-step reward")
w = 10
smooth = np.convolve(rewards, np.ones(w) / w, mode="valid")
ax0.plot(np.arange(w - 1, len(rewards)), smooth, lw=2.0, color="#C44E52",
         label=f"{w}-step moving average")
ax0.set_ylabel("reward")
ax0.set_title(f"{cfg.task} — random-action rollout ({len(rewards)} steps)")
ax0.legend(frameon=False, loc="upper right")
ax0.spines[["top", "right"]].set_visible(False)

ax1.plot(np.cumsum(rewards), lw=2.0, color="#55A868")
ax1.set_ylabel("cumulative\nreward")
ax1.set_xlabel("environment step")
ax1.spines[["top", "right"]].set_visible(False)

fig.savefig("rollout_reward.png", dpi=150)
plt.show()

### Rendered frames

`env.render()` walks the sub-environments and concatenates their frames
horizontally, so with `num_envs=1` we get a single `H × W × 3` image.

In [ ]:
n_show = 10
idx = np.linspace(0, len(frames) - 1, n_show).astype(int)

fig, axes = plt.subplots(2, 5, figsize=(12, 5.2), layout="constrained")
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(frames[i])
    ax.set_title(f"step {i * RENDER_EVERY}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"{cfg.task} — EGL offscreen renders along a random rollout",
             fontsize=12)
fig.savefig("rollout_frames.png", dpi=140)
plt.show()

In [ ]:
import imageio.v2 as imageio

imageio.mimsave("rollout.mp4", [np.ascontiguousarray(f) for f in frames], fps=15)
imageio.mimsave("rollout.gif", [np.ascontiguousarray(f[::2, ::2]) for f in frames],
                fps=15, loop=0)
print("wrote rollout.mp4 and rollout.gif "
      f"({len(frames)} frames, {frames[0].shape[1]}x{frames[0].shape[0]})")

env.close()

## 6. One task per domain

The point of MMBench is breadth, so the install is only healthy if *every*
simulator backend constructs. This builds one environment per domain, steps it
once, and renders a frame — a direct check on MuJoCo, SAPIEN (ManiSkill),
Meta-World, Gym-MuJoCo, OGBench, RoboDesk, pygame, ALE and Box2D.

**Physics and rendering are checked separately, on purpose.** MuJoCo-backed
domains render through EGL, but ManiSkill renders through **SAPIEN/Vulkan**,
which is a different graphics stack with different requirements. On this
cluster's *login* node Vulkan device creation fails
(`vk::Device::waitForFences: ErrorDeviceLost`) even with the NVIDIA ICD pinned,
because the login GPUs are shared and not available for graphics contexts.
ManiSkill *physics* is unaffected, so state-based training works fine — only
`obs=rgb` and video capture for `ms-*` tasks need a `c23g` allocation. The cell
below therefore reports `step` and `render` status independently rather than
failing.

In [ ]:
DOMAIN_SAMPLES = {
    "DMControl":  "walker-walk",
    "Meta-World": "mw-reach",
    "ManiSkill":  "ms-pick-cube",
    "Gym-MuJoCo": "mujoco-hopper",
    "OGBench":    "og-point-maze",
    "RoboDesk":   "rd-push-red",
    "MiniArcade": "pygame-pong",
    "Atari":      "atari-pong",
    "Box2D":      "bipedal-walker-flat",
}

gallery, summary = {}, []
for domain, task in DOMAIN_SAMPLES.items():
    c = build_cfg(task)
    e = make_env(c)
    row = {"domain": domain, "task": task}
    try:
        o, i = e.reset()
        o, r, term, trunc, i = e.step(e.rand_act())
        row.update(step="ok", obs_dim=int(o.shape[-1]),
                   act_dim=int(c.action_dims[0]),
                   ep_len=int(e.max_episode_steps),
                   reward=round(float(r.mean()), 4))
    except Exception as exc:
        row.update(step=type(exc).__name__, obs_dim=None, act_dim=None,
                   ep_len=None, reward=None)

    # Rendering is a separate capability from physics -- see note above.
    try:
        f = e.render(width=192, height=192)
        f = None if f is None else np.asarray(f)
        row["render"] = "ok" if f is not None else "none"
        note = ""
    except Exception as exc:
        f, note = None, type(exc).__name__
        row["render"] = note

    gallery[domain] = (task, f, note)
    summary.append(row)
    try:
        e.close()
    except Exception:
        pass
    print(f"  {domain:<11} {task:<22} step={row['step']:<12} render={row['render']}")

import pandas as pd
df = pd.DataFrame(summary).set_index("domain")
df.to_csv("domain_summary.csv")
df

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(10.5, 11), layout="constrained")
for ax, (domain, (task, frame, note)) in zip(axes.ravel(), gallery.items()):
    if frame is None:
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, color="#EAEAF2"))
        ax.text(0.5, 0.55, "render unavailable", ha="center", va="center",
                fontsize=10, color="#555")
        ax.text(0.5, 0.42, note or "no frame", ha="center", va="center",
                fontsize=8, color="#888", style="italic")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    else:
        ax.imshow(frame)
    ax.set_title(f"{domain}\n{task}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("MMBench — one environment per domain, first step after reset",
             fontsize=13)
fig.savefig("domain_gallery.png", dpi=140)
plt.show()

## Where to go next

```bash
cd ~/newt/tdmpc2
python train.py model_size=B task=walker-walk      # 5M single-task agent
python train.py                                     # 20M agent, all 200 tasks
python train.py obs=rgb                             # state + RGB observations
python download_checkpoints.py --filename "walker-walk" --cache-dir=./checkpoints
python train.py checkpoint=./checkpoints/<file>.pt   # evaluate / finetune
```

Multitask checkpoints are prefixed `soup`; single-task checkpoints need
`model_size=B`. Training is GPU work — submit it to `c23g` rather than running
it on the login node, and remember the venv lives on node-local `/tmp`, so a
job script has to unpack it first (see `SETUP_NOTES.md`).